# 02 - Annotation representation and equivalence

The same contribution as a plain discourse triple and as an `oa:Annotation`.
Both must yield the same discourse edge. Local only.

In [7]:
# Setup. Local only: signs with the local profile, never publishes.
import sys, os, subprocess, glob
try:
    import nanopub, rdflib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nanopub", "rdflib"])
sys.path.insert(0, os.getcwd())
try:
    import na_nanopub as na
except ModuleNotFoundError:
    hits = glob.glob(os.path.join(os.getcwd(), "**", "na_nanopub.py"), recursive=True)
    if hits:
        sys.path.insert(0, os.path.dirname(hits[0]))
    import na_nanopub as na
import nanopub as _np
print("nanopub", _np.__version__, "-- ready, signing locally (no network)")

nanopub 2.1.0 -- ready, signing locally (no network)


## Root claim

In [8]:
root = na.make(
    '''  sub:claim a schema:Statement ;
    rdf:value  "Certain gut microbiota profiles have stronger mRNA vaccine responses." .''',
    attributed_to="orcid:0000-0001-alvarez-b", nanopub_type="schema:Statement", introduces="sub:claim", name="02-root")
na.show(root, "root")
claim = root.source_uri + "/claim"

root           https://w3id.org/np/RAiXk9OWwlpkcwP6WPbrNu-kT2YY-Mfx9VpiDsXJCV1pY   [Statement]


## Direct representation

In [9]:
direct = na.make(
    f'''  sub:reason a schema:Statement ;
    rdf:value      "Their SCFA data is compelling." ;
    cito:supports  <{claim}> .''',
    attributed_to="orcid:0000-0002-wang-p", nanopub_type="cito:supports", introduces="sub:reason", name="02-direct")
na.show(direct, "direct")

direct         https://w3id.org/np/RA3cshVUVSLm5fKtxGV9udgBL2AbclxnXMo5JUlaAIPr0   [supports]


## Annotation representation

Two content nodes: `sub:reason` carries the discourse relation, `sub:annot`
records what it is attached to (`oa:hasTarget`).

In [10]:
annot = na.make(
    f'''  sub:reason a schema:Statement ;
    rdf:value      "Their SCFA data is compelling." ;
    cito:supports  <{claim}> .

  sub:annot a oa:Annotation ;
    oa:hasBody     sub:reason ;
    oa:hasTarget   <{root.source_uri}> ;
    oa:motivatedBy oa:assessing .''',
    attributed_to="orcid:0000-0002-wang-p", nanopub_type="cito:supports", introduces="sub:reason", name="02-annotation")
na.show(annot, "annotation")

annotation     https://w3id.org/np/RAnSMPHof5PTa6zyHImduqKfWLgpWCaSKX5BSIX4_qtww   [supports]


## Equivalence check

In [11]:
for label, np in [("direct", direct), ("annotation", annot)]:
    ds = na.load(root, np)
    rows = na.targets(ds, claim)
    print(label, "->", [(na.localname(r['relation']), na.localname(r['contribution'])) for r in rows])

direct -> [('supports', 'reason')]
annotation -> [('supports', 'reason')]


## Querying the annotation target

In [12]:
ds = na.load(annot)
q = na.QP + '''
SELECT ?annot ?body ?value ?target ?motiv WHERE { GRAPH ?g {
  ?annot a oa:Annotation ;
    oa:hasBody ?body ; oa:hasTarget ?target ; oa:motivatedBy ?motiv .
  ?body rdf:value ?value . } }'''
for r in ds.query(q):
    print("annotation:", na.localname(r.annot))
    print("  body     :", na.localname(r.body), f'"{r.value}"')
    print("  target   :", r.target)
    print("  motivated:", na.localname(r.motiv))

annotation: annot
  body     : reason "Their SCFA data is compelling."
  target   : https://w3id.org/np/RAiXk9OWwlpkcwP6WPbrNu-kT2YY-Mfx9VpiDsXJCV1pY
  motivated: assessing
